In [1]:
import pandas as pd
import numpy as np
import nltk
import spacy
import re

from nltk import RegexpTokenizer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec
from sklearn.base import BaseEstimator, TransformerMixin
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.neural_network import MLPClassifier

from sklearn.pipeline import Pipeline
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

__1. Preparación de los textos utilizando el esquema de bolsa de palabras (BOW)l con una pesado TF-IDF. Para este paso construir un pipeline que integre las transformaciones que se consideren adecuadas.__

In [2]:
columns = ['textos','ODS']
data = pd.read_excel('Train_textosODS.xlsx', usecols=['textos','ODS'])
data.sample(5)

,textos,ODS
1250,El papel de las transferencias en las estrateg...,8
1862,Si bien los NWMP y la Estrategia para una Econ...,12
3084,Una gran parte de la ganancia en la esperanza ...,3
6476,El acceso a la justicia es a la vez un tema de...,16
5643,I. Tratados y derechos individuales 105 II. El...,16


In [3]:
print(f'data duplicada: {data.duplicated().sum()}')

data duplicada: 0


In [4]:
print(f'data vacia: {data.isna().sum()}')

data vacia: textos    0
ODS       0
dtype: int64


In [5]:
x = data['textos']
y = data['ODS']

In [6]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [7]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
nlp = spacy.load("es_core_news_sm")
stop_words = stopwords.words('spanish')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\egcv_\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\egcv_\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\egcv_\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [8]:
class TextPreprocessor(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X):

        cleaned = []

        for text in X:

            # minúsculas
            text = text.lower()

            # eliminar números
            text = re.sub(r'\d+', '', text)

            # eliminar puntuación
            text = re.sub(r'[^\w\s]', '', text)

            # tokenización
            tokens = word_tokenize(text)

            # eliminar stopwords
            tokens = [t for t in tokens if t not in stop_words]

            cleaned.append(" ".join(tokens))

        return cleaned

In [9]:
tfidf = TfidfVectorizer(
    ngram_range=(1,2),
    min_df=2,
    max_features=5000)

In [10]:
class Word2VecVectorizer(BaseEstimator, TransformerMixin):

    def __init__(self, vector_size=100):
        self.vector_size = vector_size

    def fit(self, X, y=None):

        tokenized = [text.split() for text in X]

        self.model = Word2Vec(
            sentences=tokenized,
            vector_size=self.vector_size,
            window=5,
            min_count=2
        )

        return self

    def transform(self, X):

        vectors = []

        for text in X:

            words = text.split()

            word_vectors = [
                self.model.wv[w]
                for w in words
                if w in self.model.wv
            ]

            if len(word_vectors) == 0:
                vectors.append(np.zeros(self.vector_size))
            else:
                vectors.append(np.mean(word_vectors, axis=0))

        return np.array(vectors)

In [11]:
pipeline_tfidf = Pipeline([

    ("preprocess", TextPreprocessor()),

    ("tfidf", tfidf),

    ("classifier", LogisticRegression(max_iter=500))

])

In [12]:
pipeline_w2v = Pipeline([

    ("preprocess", TextPreprocessor()),

    ("word2vec", Word2VecVectorizer(vector_size=100)),

    ("model", LogisticRegression(max_iter=500))

])

In [31]:
pipeline_w2v_nn = Pipeline([
    
    ("preprocess", TextPreprocessor()),
    
    ("word2vec", Word2VecVectorizer()),
    
    ("classifier", MLPClassifier(
        hidden_layer_sizes=(256,128),
        activation="logistic",
        solver="adam",
        max_iter=1000,
        random_state=42
    ))
])

In [32]:
pipeline_tfidf.fit(x_train, y_train)

,steps,"[('preprocess', ...), ('tfidf', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


In [33]:
pipeline_w2v.fit(x_train, y_train)

,steps,"[('preprocess', ...), ('word2vec', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,vector_size,100
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1


In [34]:
pipeline_w2v_nn.fit(x_train, y_train)

,steps,"[('preprocess', ...), ('word2vec', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,vector_size,100
,hidden_layer_sizes,"(256, ...)"
,activation,'logistic'
,solver,'adam'
,alpha,0.0001
,batch_size,'auto'
,learning_rate,'constant'


In [35]:
pred_tfidf = pipeline_tfidf.predict(x_test)

pred_w2v = pipeline_w2v.predict(x_test)

pred_nn = pipeline_w2v_nn.predict(x_test)

In [36]:
print("\nReporte TF-IDF")
print(classification_report(y_test, pred_tfidf))

print("\nReporte Word2Vec")
print(classification_report(y_test, pred_w2v))

print("\nReporte Word2Vec+NN")
print(classification_report(y_test, pred_nn))


Reporte TF-IDF
              precision    recall  f1-score   support

           1       0.86      0.86      0.86        90
           2       0.80      0.86      0.83        71
           3       0.90      0.94      0.92       178
           4       0.89      0.99      0.94       200
           5       0.94      0.93      0.94       232
           6       0.93      0.93      0.93       135
           7       0.92      0.93      0.93       164
           8       0.69      0.69      0.69        86
           9       0.83      0.82      0.83        55
          10       0.76      0.52      0.61        60
          11       0.88      0.85      0.87       143
          12       0.93      0.83      0.88        64
          13       0.83      0.81      0.82       102
          14       0.95      0.86      0.90        64
          15       0.99      0.84      0.91        81
          16       0.91      0.98      0.94       207

    accuracy                           0.89      1932
   macro a